# 13 — microRNA activity in aging cells

microRNAs (miRNAs) are ~22-nucleotide RNAs that bind messenger RNAs and lower their level. Droplet single-cell sequencing captures polyadenylated mRNA, so **mature miRNAs are not measured**. Their activity can still be read from their **targets**: if a miRNA becomes more active in old cells, its target mRNAs should fall together.

| Step | Data | Method |
| --- | --- | --- |
| miRNA → target network | OmniPath `mirnatarget` (miRTarBase, miRecords, miRDeathDB, ncRDeathDB, miR2Disease: experimentally supported), human, targets mapped to mouse orthologs | miRNA seed regions of the aging miRNAs are conserved between human and mouse |
| activity per tissue × cell type | DESeq2 statistics from 09 (mice as samples) | `decoupler.mt.ulm`, target weight −1: activity > 0 means the targets fall together |
| check against measurement | TMS **bulk** RNA (Schaum et al. 2020): 17 organs, 10 ages, ~54 mice | the primary transcript (*Mir155hg*, *Mir22hg*, *Mir17hg*, *Mir29a* …) is a measured precursor of the mature miRNA |

A priori miRNAs (chosen before running): **miR-34a** (p53 target, senescence), **miR-146a** (brake on NF-κB, inflammaging), **miR-29a/b/c** (represses collagens), **miR-21**, **miR-155** (inflammation), **miR-22**, **miR-17** (the miR-17~92 cluster).

### 1. Setup

In [ ]:
import sys, re
sys.path.append("..")          # so "src" can be imported from the notebook folder

import numpy as np
import pandas as pd
import scanpy as sc
import anndata as ad
import matplotlib.pyplot as plt
import seaborn as sns
import decoupler as dc
from scipy.stats import spearmanr
from src import aging as ag

A = ag.paths("atlas")
atlas = pd.read_csv(A.figure("atlas.csv"))
atlas["robust"] = atlas["verdict"].eq("age signal") & ~atlas["tech_confounded"].astype(bool)
atlas["row"] = atlas["tissue"] + " | " + atlas["cell_type"]
de = pd.read_csv(A.figure("pseudobulk_de.csv.gz"))
stat = de.pivot_table(index="row", columns="gene", values="stat").fillna(0)
robust = [r for r in stat.index if r in set(atlas.loc[atlas["robust"], "row"])]
tissue_of = dict(zip(atlas["row"], atlas["tissue"]))
comp = atlas.set_index("row")["compartment"]
PRIORI = ["miR-34a", "miR-146a", "miR-29a", "miR-29b", "miR-29c", "miR-21", "miR-155", "miR-22", "miR-17"]

### 2. Build the miRNA → mouse target network

Names in the source databases mix old and new styles (`hsa-miR-146;hsa-miR-146a`, `hsa-miR-29b`, `hsa-miR-21-5p`). Each is reduced to one name: `hsa-` and the `-5p` arm suffix are dropped (the 5p arm is the usual mature form), `-3p` and passenger (`*`) strands are kept apart, and a list like `miR-146;miR-146a` takes its most specific entry. Human target symbols are mapped to mouse orthologs with `decoupler.op.translate` (HCOP).

In [ ]:
raw = pd.read_csv(ag.ROOT / "data" / "external" / "decoupler" / "omnipath_mirna_human.tsv", sep="\t")

def mirna_name(s):
    name = s.split(";")[-1].replace("hsa-", "")
    return name[:-3] if name.endswith("-5p") else name

net = pd.DataFrame({"source": raw["source_genesymbol"].map(mirna_name), "target": raw["target_genesymbol"]}).drop_duplicates()
net = dc.op.translate(net, columns=["target"], target_organism="mouse")
net = net.drop_duplicates(["source", "target"]).assign(weight=-1.0)       # a miRNA lowers its targets
net.to_csv(ag.ROOT / "data" / "external" / "decoupler" / "mirna_targets_mouse.csv", index=False)
size = net.groupby("source").size()
print(len(net), "pairs,", net["source"].nunique(), "miRNAs,", (size >= 10).sum(), "with >= 10 targets")
size.reindex(PRIORI)

### 3. Activity per tissue × cell type

Activity of every miRNA with at least 10 targets present, in all 55 rows of 09. `tissues_up`: tissues with a robust row at activity > 2 (targets fall together in old cells).

In [ ]:
act, act_p = dc.mt.ulm(data=stat, net=net, tmin=10)
act.round(3).to_csv(A.figure("mirna_activity.csv"))

def shared(es, rows, thr=2.0):
    e = es.loc[rows]
    return pd.DataFrame({"mean_activity": e.mean(), "rows_up": (e > thr).sum(), "rows_down": (e < -thr).sum(),
                         "tissues_up": (e > thr).groupby(e.index.map(tissue_of)).any().sum(),
                         "tissues_down": (e < -thr).groupby(e.index.map(tissue_of)).any().sum()})

sh = shared(act, robust).sort_values("mean_activity", ascending=False)
sh.to_csv(A.figure("mirna_shared.csv"))
print("a priori miRNAs, robust rows:")
print(sh.reindex([m for m in PRIORI if m in sh.index]).round(2))
print("\nmost consistently more active in old cells (>= 4 tissues):")
sh[sh["tissues_up"] >= 4].head(15).round(2)

### 4. Is the miRNA signal just the inflammation signal again?

Many miRNA targets are inflammatory or matrix genes. If a miRNA's "activity" only restates genes already known to move (12), it adds nothing. Two checks: activity recomputed **without collagen and matrix genes** (`Col*`, `Sparc`, `Fn1`, `Eln`, `Lox*`, `Mfap*`), and its correlation across rows with the NF-κB activity of 12.

In [ ]:
ecm = stat.columns[stat.columns.str.match(r"^(Col\d|Sparc$|Fn1$|Eln$|Lox|Mfap|Fbn|Lum$|Dcn$|Bgn$)")]
act_noecm, _ = dc.mt.ulm(data=stat.drop(columns=ecm), net=net, tmin=10)
tf = pd.read_csv(A.figure("mechanism_tf_activity.csv"), index_col=0)
keep = [m for m in PRIORI if m in act.columns]
pd.DataFrame({"robust_mean": act.loc[robust, keep].mean(),
              "robust_mean_no_matrix": act_noecm.loc[robust, keep].mean(),
              "stromal_rows_mean": act.loc[[r for r in robust if comp[r] == "stromal"], keep].mean(),
              "stromal_no_matrix": act_noecm.loc[[r for r in robust if comp[r] == "stromal"], keep].mean(),
              "rho_with_NFkB_activity": [spearmanr(act.loc[robust, m], tf.loc[robust, "Nfkb1"])[0] for m in keep]}).round(2)

### 5. Check against measurement: primary miRNA transcripts in bulk RNA

The TMS bulk atlas (Schaum et al. 2020) sequenced 17 organs from ~54 mice at 10 ages (1–27 months), and its sequencing plates are balanced across ages, so plate is not tied to age there. A miRNA's primary transcript (pri-miRNA, e.g. *Mir155hg*) is captured by total RNA sequencing. Per organ: Spearman ρ of the precursor with age over mice, and the **inferred** activity in that organ computed the same way as above, from every gene's ρ with age. If inference works, organs where the precursor rises should be organs where activity rises.

In [ ]:
bulk = ad.read_h5ad(ag.ROOT / "data" / "external" / "190304_maca_bulk.h5ad")
bulk = bulk[bulk.obs["Organ"].astype(str).ne("nan").to_numpy()]
lib = np.asarray(bulk.X.sum(axis=1)).ravel()
bulk = bulk[lib > 1e6].copy()                                  # drops failed libraries (min was 34 reads)
sc.pp.normalize_total(bulk, target_sum=1e6)
sc.pp.log1p(bulk)
print(bulk.shape, "samples x genes;", bulk.obs["Organ"].nunique(), "organs")

HOST = {"miR-155": "Mir155hg", "miR-22": "Mir22hg", "miR-17": "Mir17hg", "miR-29a": "Mir29a", "miR-29b": "Mir29b-1",
        "miR-21": "Mir21a", "miR-34a": "Mir34a", "miR-146a": "Mir146", "miR-29c": "Mir29c"}
HOST = {m: g for m, g in HOST.items() if g in bulk.var_names}
print("precursors present:", HOST)

rho_rows = {}
for organ, idx in bulk.obs.groupby("Organ", observed=True).indices.items():
    sub = bulk[idx]
    X = sub.X.toarray() if hasattr(sub.X, "toarray") else np.asarray(sub.X)
    expressed = X.mean(axis=0) > 1
    r = pd.DataFrame(X[:, expressed], columns=sub.var_names[expressed]).rank().corrwith(
        pd.Series(sub.obs["Age"].to_numpy()).rank())
    rho_rows[organ] = r
rho = pd.DataFrame(rho_rows).T.fillna(0)                       # organs x genes: Spearman rho with age
z = rho * np.sqrt(bulk.obs.groupby("Organ", observed=True).size().reindex(rho.index) - 3).to_numpy()[:, None]
bulk_act, _ = dc.mt.ulm(data=z, net=net, tmin=10)
check = []
for m, g in HOST.items():
    if m in bulk_act.columns and g in rho.columns:
        r, p = spearmanr(rho[g], bulk_act[m])
        check.append({"miRNA": m, "precursor": g, "organs": len(rho), "precursor_rho_median": rho[g].median(),
                      "activity_median": bulk_act[m].median(), "organs_agree_rho": r, "p": p})
check = pd.DataFrame(check)
check.round(3).to_csv(A.figure("mirna_bulk_check.csv"), index=False)
check.round(3)

### 6. Precursors over age, organ by organ

The measured precursor trend of the a priori miRNAs in each organ (Spearman ρ with age; 10 ages, ~5 mice each).

In [ ]:
show = rho[[g for g in HOST.values() if g in rho.columns]]
fig, ax = plt.subplots(figsize=(0.7 * show.shape[1] + 2, 0.32 * show.shape[0] + 1))
sns.heatmap(show, cmap="RdBu_r", center=0, vmin=-0.8, vmax=0.8, annot=True, fmt=".2f", annot_kws={"size": 6}, ax=ax,
            cbar_kws={"label": "Spearman rho with age (bulk)"})
ax.set_title("miRNA precursors across age, per organ")
fig.savefig(A.figure("mirna_precursors_bulk.png"), dpi=200, bbox_inches="tight")
show.round(2)

### 7. Why the bulk check covers only two miRNAs

The filter in section 5 keeps genes with a mean log-CPM above 1, which removes most primary miRNA transcripts. Without the filter: how often is each precursor detected at all, and how do collagens, the main miR-29 targets, change with age in bulk?

In [ ]:
genes = [g for g in ["Mir29a", "Mir29b-1", "Mir29c", "Mir155hg", "Mir34a", "Mir146", "Mir21a", "Col1a1", "Col3a1"] if g in bulk.var_names]
raw = ad.read_h5ad(ag.ROOT / "data" / "external" / "190304_maca_bulk.h5ad")
raw = raw[raw.obs_names.isin(bulk.obs_names)]
Xr = raw[:, genes].X
Xr = pd.DataFrame(Xr.toarray() if hasattr(Xr, "toarray") else np.asarray(Xr), columns=genes, index=raw.obs_names)
print("fraction of samples with any read:", (Xr > 0).mean().round(2).to_dict())
age = raw.obs["Age"]
pd.DataFrame({o: {g: spearmanr(age[idx], Xr.loc[idx, g])[0] for g in genes}
              for o, idx in raw.obs.groupby("Organ", observed=True).groups.items()}).T.round(2)

### Findings

Network: 11,333 experimentally supported miRNA → target pairs (human, mapped to mouse orthologs), 278 miRNAs with ≥ 10 targets. Activity from the 09 statistics (mice as samples), 30 robust rows (22 before sex genes left the verdicts).

- **miR-29 is more active in aging stroma.** Its targets fall together in robust rows of 6–7 tissues (miR-29a 17 rows, miR-29c 17, miR-29b 16), most strongly in stromal rows (activity 4.8–6.4). Collagens are miR-29 targets, but the signal is not only them counted twice: without collagen and matrix genes, stromal activity stays at 1.9–3.6. It is independent of the inflammatory axis (correlation with NF-κB activity −0.2 to 0). This fits the known rise of miR-29 with age in mouse tissues (Ugalde et al. 2011) and its role in repressing collagen. It remains an **inference from target genes**: the network is human-derived, and the mature miRNA was not measured.
- **The inflammation miRNAs cannot be read this way.** miR-34a, miR-146a and miR-155 come out slightly *less* active (−0.5 to −1.0), but their targets are largely inflammatory genes that NF-κB drives up in old cells (activity correlates −0.47 to −0.59 with NF-κB). Rising targets here reflect transcription, not a quieter miRNA.
- **The network's human origin shows.** The top unbiased hit, miR-767, is to my knowledge human-specific; its "activity" is a set of mouse orthologs, not a mouse miRNA. Only conserved miRNAs are interpreted.
- **The bulk precursor check is not possible with this library.** The bulk atlas captures polyadenylated RNA: *Mir29a* has any read in 27% of samples, *Mir29c* and *Mir21a* in none. Only *Mir155hg* is measurable (83%), rising with age in lung (ρ 0.43), bone (0.42) and fat. For *Mir22hg* and *Mir17hg* the inferred and measured organ trends do not agree (ρ −0.23, −0.42). A real check needs measured mature miRNAs, such as the small-RNA atlas of the same cohort (Wagner et al. 2024).
- **Bulk does confirm the target side:** collagen falls with age in nearly every organ (*Col1a1* ρ −0.84 limb muscle, −0.69 brown fat, −0.62 heart, −0.61 lung) across 10 ages, with plates balanced over age.